# 📊 O*NET Dataset — Exploratory Data Analysis (EDA)
### Understanding the raw data before any ML

This notebook is **purely for understanding the datasets**.  
No models, no predictions — just raw data exploration.

**Datasets explored:**
1. Occupation Data — what jobs exist
2. Skills — what skills each job needs
3. Abilities — cognitive/physical abilities per job
4. Knowledge — domain knowledge per job
5. Interests (RIASEC) — personality-fit scores per job
6. Work Activities — day-to-day tasks per job
7. Work Styles — personality traits per job
8. Job Zones — education/experience levels
9. Task Statements — free-text task descriptions


## 0. Setup & Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

# ── Point this to your data/raw/ folder ──────────────────────────────────────
import sys
sys.path.append("..")
from config import DATA_RAW, ONET_FILES, RIASEC_CODES

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:.3f}".format)
plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.spines.top"]   = False
plt.rcParams["axes.spines.right"] = False

print("✅ Setup complete")
print(f"📁 Data path: {DATA_RAW}")


✅ Setup complete
📁 Data path: D:\adity\career_recommendation_system\data\raw


---
## 1. Load All 9 Raw O*NET Files

Each file is **long-format**: one row = one (occupation, element, scale) combination.  
We'll look at shape, columns, and sample rows for each.


In [2]:
def load(key):
    path = DATA_RAW / ONET_FILES[key]
    df = pd.read_excel(path)
    df.columns = [c.strip() for c in df.columns]
    return df

occ  = load("occupation")
sk   = load("skills")
ab   = load("abilities")
kn   = load("knowledge")
intr = load("interests")
wa   = load("work_activities")
ws   = load("work_styles")
jz   = load("job_zones")
ts   = load("tasks")

datasets = {
    "Occupation Data":    occ,
    "Skills":             sk,
    "Abilities":          ab,
    "Knowledge":          kn,
    "Interests":          intr,
    "Work Activities":    wa,
    "Work Styles":        ws,
    "Job Zones":          jz,
    "Task Statements":    ts,
}

print(f"{'Dataset':<25} {'Rows':>8} {'Cols':>6}  Columns")
print("-"*80)
for name, df in datasets.items():
    cols = ", ".join(df.columns[:5].tolist()) + ("..." if len(df.columns)>5 else "")
    print(f"  {name:<23} {df.shape[0]:>8,} {df.shape[1]:>6}  {cols}")


Dataset                       Rows   Cols  Columns
--------------------------------------------------------------------------------
  Occupation Data            1,016      3  O*NET-SOC Code, Title, Description
  Skills                    62,580     15  O*NET-SOC Code, Title, Element ID, Element Name, Scale ID...
  Abilities                 92,976     15  O*NET-SOC Code, Title, Element ID, Element Name, Scale ID...
  Knowledge                 59,004     15  O*NET-SOC Code, Title, Element ID, Element Name, Scale ID...
  Interests                  8,307      9  O*NET-SOC Code, Title, Element ID, Element Name, Scale ID...
  Work Activities           73,308     15  O*NET-SOC Code, Title, Element ID, Element Name, Scale ID...
  Work Styles               37,422      9  O*NET-SOC Code, Title, Element ID, Element Name, Scale ID...
  Job Zones                    923      5  O*NET-SOC Code, Title, Job Zone, Date, Domain Source
  Task Statements           18,796      8  O*NET-SOC Code, Title, Task

---
## 2. Occupation Data
The master list of all occupations with their SOC codes and descriptions.  
**This is the backbone** — every other file links back here via `O*NET-SOC Code`.


In [3]:
print(f"Total occupations: {occ.shape[0]:,}")
print(f"\nSample rows:")
occ[["O*NET-SOC Code", "Title", "Description"]].head(8)


Total occupations: 1,016

Sample rows:


,O*NET-SOC Code,Title,Description
0,11-1011.00,Chief Executives,Determine and formulate policies and provide o...
1,11-1011.03,Chief Sustainability Officers,"Communicate and coordinate with management, sh..."
2,11-1021.00,General and Operations Managers,"Plan, direct, or coordinate the operations of ..."
3,11-1031.00,Legislators,"Develop, introduce, or enact laws and statutes..."
4,11-2011.00,Advertising and Promotions Managers,"Plan, direct, or coordinate advertising polici..."
5,11-2021.00,Marketing Managers,"Plan, direct, or coordinate marketing policies..."
6,11-2022.00,Sales Managers,"Plan, direct, or coordinate the actual distrib..."
7,11-2032.00,Public Relations Managers,"Plan, direct, or coordinate activities designe..."


In [4]:
# How long are the descriptions?
occ["desc_length"] = occ["Description"].str.len()
fig, ax = plt.subplots(figsize=(9,4))
ax.hist(occ["desc_length"].dropna(), bins=40, color="#3498db", alpha=0.85, edgecolor="white")
ax.set_title("Occupation Description Length Distribution", fontweight="bold")
ax.set_xlabel("Character count")
ax.set_ylabel("Number of occupations")
ax.axvline(occ["desc_length"].mean(), color="red", linestyle="--",
           label=f"Mean = {occ['desc_length'].mean():.0f} chars")
ax.legend()
plt.tight_layout(); plt.show()


---
## 3. Skills Dataset
**What it contains:** Each row = one skill score for one occupation.  
Format: `O*NET-SOC Code | Element Name | Scale Name | Data Value`

**Scale types:**
- `Importance` — how important is this skill (1–5)
- `Level` — how advanced the skill needs to be (0–7)

We use **Level** scores as features (more discriminating than Importance).


In [5]:
print(f"Shape: {sk.shape}")
print(f"\nColumns: {sk.columns.tolist()}")
print(f"\nSample rows:")
display(sk.head(8))


Shape: (62580, 15)

Columns: ['O*NET-SOC Code', 'Title', 'Element ID', 'Element Name', 'Scale ID', 'Scale Name', 'Data Value', 'N', 'Standard Error', 'Lower CI Bound', 'Upper CI Bound', 'Recommend Suppress', 'Not Relevant', 'Date', 'Domain Source']

Sample rows:


,O*NET-SOC Code,Title,Element ID,Element Name,Scale ID,Scale Name,Data Value,N,Standard Error,Lower CI Bound,Upper CI Bound,Recommend Suppress,Not Relevant,Date,Domain Source
0,11-1011.00,Chief Executives,2.A.1.a,Reading Comprehension,IM,Importance,4.120,8,0.125,3.880,4.370,N,NaN,08/2023,Analyst
1,11-1011.00,Chief Executives,2.A.1.a,Reading Comprehension,LV,Level,4.620,8,0.183,4.266,4.984,N,N,08/2023,Analyst
2,11-1011.00,Chief Executives,2.A.1.b,Active Listening,IM,Importance,4.000,8,0.000,4.000,4.000,N,NaN,08/2023,Analyst
3,11-1011.00,Chief Executives,2.A.1.b,Active Listening,LV,Level,4.750,8,0.164,4.429,5.071,N,N,08/2023,Analyst
4,11-1011.00,Chief Executives,2.A.1.c,Writing,IM,Importance,4.120,8,0.125,3.880,4.370,N,NaN,08/2023,Analyst
5,11-1011.00,Chief Executives,2.A.1.c,Writing,LV,Level,4.380,8,0.183,4.016,4.734,N,N,08/2023,Analyst
6,11-1011.00,Chief Executives,2.A.1.d,Speaking,IM,Importance,4.250,8,0.164,3.929,4.571,N,NaN,08/2023,Analyst
7,11-1011.00,Chief Executives,2.A.1.d,Speaking,LV,Level,4.750,8,0.164,4.429,5.071,N,N,08/2023,Analyst


In [6]:
print(f"Unique skills (Element Names): {sk['Element Name'].nunique()}")
print(f"Unique occupations:            {sk['O*NET-SOC Code'].nunique()}")
print(f"Scale types:                   {sk['Scale Name'].unique().tolist()}")
print(f"\nData Value stats (Level scale):")
level_sk = sk[sk["Scale Name"]=="Level"]["Data Value"]
print(level_sk.describe().round(3))


Unique skills (Element Names): 35
Unique occupations:            894
Scale types:                   ['Importance', 'Level']

Data Value stats (Level scale):
count   31290.000
mean        2.376
std         1.268
min         0.000
25%         1.500
50%         2.620
75%         3.250
max         6.000
Name: Data Value, dtype: float64


In [7]:
# Top 20 most variable skills (highest std = most discriminating)
level_data = sk[sk["Scale Name"]=="Level"].copy()
skill_stats = (
    level_data.groupby("Element Name")["Data Value"]
    .agg(["mean","std","min","max"])
    .sort_values("std", ascending=False)
    .head(20)
)

fig, ax = plt.subplots(figsize=(11, 5))
ax.barh(skill_stats.index[::-1], skill_stats["std"][::-1], color="#e74c3c", alpha=0.85)
ax.set_title("Top 20 Skills by Standard Deviation (Most Discriminating)", fontweight="bold")
ax.set_xlabel("Std Dev of Level Score across Occupations")
ax.grid(True, alpha=0.3, axis="x")
plt.tight_layout(); plt.show()
print(skill_stats.round(3))


                                   mean   std   min   max
Element Name                                             
Science                           1.447 1.399 0.000 5.750
Repairing                         1.021 1.238 0.000 4.380
Equipment Maintenance             1.054 1.231 0.000 5.380
Operation and Control             1.756 1.229 0.000 5.620
Troubleshooting                   1.554 1.145 0.000 4.380
Operations Analysis               1.647 1.083 0.000 5.000
Equipment Selection               1.075 1.030 0.000 4.120
Operations Monitoring             2.259 1.001 0.000 4.880
Quality Control Analysis          2.159 0.996 0.000 4.250
Management of Financial Resources 1.343 0.910 0.000 5.250
Mathematics                       2.558 0.897 0.000 6.000
Systems Evaluation                2.637 0.861 0.120 5.000
Writing                           3.311 0.852 1.000 5.750
Reading Comprehension             3.653 0.843 1.620 5.880
Programming                       0.831 0.819 0.000 4.880
Systems Analys

In [8]:
# Distribution of a specific skill across all occupations
skill_name = "Critical Thinking"
skill_dist = level_data[level_data["Element Name"]==skill_name]["Data Value"]

fig, ax = plt.subplots(figsize=(9,4))
ax.hist(skill_dist.dropna(), bins=30, color="#2ecc71", alpha=0.85, edgecolor="white")
ax.set_title(f"Distribution of '{skill_name}' Level Score Across All Occupations",
             fontweight="bold")
ax.set_xlabel("Level Score (0–7)")
ax.set_ylabel("Number of Occupations")
ax.axvline(skill_dist.mean(), color="red", linestyle="--",
           label=f"Mean = {skill_dist.mean():.2f}")
ax.legend()
plt.tight_layout(); plt.show()


---
## 4. Abilities Dataset
Similar structure to Skills but covers **cognitive and physical abilities**:  
e.g. Deductive Reasoning, Spatial Orientation, Manual Dexterity, Hearing Sensitivity.


In [9]:
print(f"Shape: {ab.shape}")
print(f"Unique abilities: {ab['Element Name'].nunique()}")
print(f"\nAll ability names:")
for i, name in enumerate(sorted(ab["Element Name"].unique()), 1):
    print(f"  {i:2d}. {name}")


Shape: (92976, 15)
Unique abilities: 52

All ability names:
   1. Arm-Hand Steadiness
   2. Auditory Attention
   3. Category Flexibility
   4. Control Precision
   5. Deductive Reasoning
   6. Depth Perception
   7. Dynamic Flexibility
   8. Dynamic Strength
   9. Explosive Strength
  10. Extent Flexibility
  11. Far Vision
  12. Finger Dexterity
  13. Flexibility of Closure
  14. Fluency of Ideas
  15. Glare Sensitivity
  16. Gross Body Coordination
  17. Gross Body Equilibrium
  18. Hearing Sensitivity
  19. Inductive Reasoning
  20. Information Ordering
  21. Manual Dexterity
  22. Mathematical Reasoning
  23. Memorization
  24. Multilimb Coordination
  25. Near Vision
  26. Night Vision
  27. Number Facility
  28. Oral Comprehension
  29. Oral Expression
  30. Originality
  31. Perceptual Speed
  32. Peripheral Vision
  33. Problem Sensitivity
  34. Rate Control
  35. Reaction Time
  36. Response Orientation
  37. Selective Attention
  38. Sound Localization
  39. Spatial Orientat

In [10]:
# Heatmap: mean ability score per ability (importance scale)
imp_ab = ab[ab["Scale Name"]=="Importance"].groupby("Element Name")["Data Value"].mean()
imp_ab = imp_ab.sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(11, 6))
bars = ax.barh(imp_ab.index[::-1], imp_ab.values[::-1],
               color=plt.cm.Blues(np.linspace(0.4, 0.9, len(imp_ab))))
ax.set_title("Mean Importance of Each Ability Across All Occupations", fontweight="bold")
ax.set_xlabel("Mean Importance Score (1–5)")
ax.grid(True, alpha=0.3, axis="x")
plt.tight_layout(); plt.show()


---
## 5. Knowledge Dataset
Domain knowledge areas: e.g. Mathematics, Medicine, Law, Engineering, Psychology.  
High missingness (10%) because knowledge domains are **highly occupation-specific**.


In [11]:
print(f"Shape: {kn.shape}")
print(f"Unique knowledge domains: {kn['Element Name'].nunique()}")

# Missing value breakdown
missing = kn.isnull().sum()
missing_pct = (missing / len(kn) * 100).round(2)
print(f"\nMissing values: {missing.sum():,} ({missing_pct.sum():.1f}% of cells)")
print(f"\nAll knowledge domains:")
for i, name in enumerate(sorted(kn["Element Name"].unique()), 1):
    print(f"  {i:2d}. {name}")


Shape: (59004, 15)
Unique knowledge domains: 33

Missing values: 90,608 (153.6% of cells)

All knowledge domains:
   1. Administration and Management
   2. Administrative
   3. Biology
   4. Building and Construction
   5. Chemistry
   6. Communications and Media
   7. Computers and Electronics
   8. Customer and Personal Service
   9. Design
  10. Economics and Accounting
  11. Education and Training
  12. Engineering and Technology
  13. English Language
  14. Fine Arts
  15. Food Production
  16. Foreign Language
  17. Geography
  18. History and Archeology
  19. Law and Government
  20. Mathematics
  21. Mechanical
  22. Medicine and Dentistry
  23. Personnel and Human Resources
  24. Philosophy and Theology
  25. Physics
  26. Production and Processing
  27. Psychology
  28. Public Safety and Security
  29. Sales and Marketing
  30. Sociology and Anthropology
  31. Telecommunications
  32. Therapy and Counseling
  33. Transportation


In [12]:
# Which knowledge domains vary most across occupations?
lv_kn = kn[kn["Scale Name"]=="Level"].groupby("Element Name")["Data Value"].std()
lv_kn = lv_kn.sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10,5))
ax.barh(lv_kn.index[::-1], lv_kn.values[::-1], color="#9b59b6", alpha=0.85)
ax.set_title("Top 15 Knowledge Domains by Variance (Most Career-Discriminating)",
             fontweight="bold")
ax.set_xlabel("Std Dev of Level Score")
ax.grid(True, alpha=0.3, axis="x")
plt.tight_layout(); plt.show()


---
## 6. Interests Dataset — RIASEC Scores ⭐ (Target Variable)

This is the **most important dataset** for our ML system.

**RIASEC** = 6 personality/career types developed by psychologist John Holland:
| Code | Type | Description |
|------|------|-------------|
| R | Realistic | Hands-on, mechanical, physical |
| I | Investigative | Research, analysis, science |
| A | Artistic | Creative, expressive, design |
| S | Social | Helping, teaching, counseling |
| E | Enterprising | Leadership, business, persuasion |
| C | Conventional | Data, organizing, administration |

Each occupation has a score for each type. We normalize so they sum to 1 → **probability distribution**.  
This becomes our multi-output regression target.


In [13]:
print(f"Shape: {intr.shape}")
print(f"Unique interest elements: {intr['Element Name'].nunique()}")
print(f"\nAll element names (we only use the 6 top-level RIASEC):")
print(intr["Element Name"].unique().tolist())


Shape: (8307, 9)
Unique interest elements: 9

All element names (we only use the 6 top-level RIASEC):
['Realistic', 'Investigative', 'Artistic', 'Social', 'Enterprising', 'Conventional', 'First Interest High-Point', 'Second Interest High-Point', 'Third Interest High-Point']


In [14]:
# Extract top-level RIASEC scores
riasec_raw = intr[intr["Element Name"].isin(RIASEC_CODES)].copy()
riasec_raw["Data Value"] = pd.to_numeric(riasec_raw["Data Value"], errors="coerce")

riasec_pivot = riasec_raw.pivot_table(
    index="O*NET-SOC Code", columns="Element Name",
    values="Data Value", aggfunc="mean"
)[RIASEC_CODES]

# Normalize → probability distribution
riasec_norm = riasec_pivot.div(riasec_pivot.sum(axis=1), axis=0).fillna(0)

print(f"RIASEC matrix shape: {riasec_norm.shape}")
print(f"\nFirst 5 occupations (normalized RIASEC scores):")
# Show with titles
titles = occ.set_index("O*NET-SOC Code")["Title"]
sample = riasec_norm.head(5).copy()
sample.insert(0, "Occupation", titles.reindex(sample.index).values)
display(sample.round(3))


RIASEC matrix shape: (923, 6)

First 5 occupations (normalized RIASEC scores):


Element Name,Occupation,Realistic,Investigative,Artistic,Social,Enterprising,Conventional
O*NET-SOC Code,,,,,,,
11-1011.00,Chief Executives,0.057,0.139,0.098,0.161,0.317,0.227
11-1011.03,Chief Sustainability Officers,0.085,0.199,0.103,0.148,0.278,0.187
11-1021.00,General and Operations Managers,0.102,0.110,0.060,0.157,0.324,0.247
11-1031.00,Legislators,0.075,0.164,0.132,0.181,0.270,0.177
11-2011.00,Advertising and Promotions Managers,0.047,0.080,0.183,0.149,0.331,0.210


In [15]:
# Distribution of each RIASEC score
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
colors = ["#e74c3c","#3498db","#2ecc71","#f39c12","#9b59b6","#1abc9c"]

for i, (code, color) in enumerate(zip(RIASEC_CODES, colors)):
    axes[i].hist(riasec_norm[code], bins=40, color=color, alpha=0.85, edgecolor="white")
    axes[i].set_title(f"{code}", fontweight="bold")
    axes[i].set_xlabel("Normalized score (0–1)")
    axes[i].set_ylabel("Frequency")
    axes[i].axvline(riasec_norm[code].mean(), color="black", linestyle="--",
                    label=f"Mean={riasec_norm[code].mean():.3f}")
    axes[i].legend(fontsize=9)
    axes[i].grid(True, alpha=0.3)

plt.suptitle("RIASEC Score Distributions Across All Occupations\n(Normalized — sums to 1 per occupation)",
             fontsize=14, fontweight="bold")
plt.tight_layout(); plt.show()


In [16]:
# Dominant RIASEC type per occupation
dominant = riasec_norm.idxmax(axis=1).value_counts()

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(dominant.index, dominant.values, color=colors, alpha=0.85, edgecolor="white")
ax.set_title("How Many Occupations Have Each RIASEC Type as Dominant?", fontweight="bold")
ax.set_xlabel("Dominant RIASEC Type")
ax.set_ylabel("Number of Occupations")
for bar, val in zip(bars, dominant.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+3,
            str(val), ha="center", fontweight="bold")
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout(); plt.show()
print(dominant)


Realistic        388
Conventional     166
Social           129
Investigative    112
Enterprising     100
Artistic          28
Name: count, dtype: int64


In [17]:
# RIASEC correlation heatmap
fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(riasec_norm.corr(), dtype=bool))
sns.heatmap(riasec_norm.corr(), annot=True, fmt=".3f", cmap="coolwarm",
            center=0, mask=mask, ax=ax, linewidths=0.5,
            annot_kws={"size":12,"weight":"bold"})
ax.set_title("Correlation Between RIASEC Types\n(Negative = mutually exclusive career profiles)",
             fontweight="bold")
plt.tight_layout(); plt.show()


In [18]:
# Show example occupations for each RIASEC type
print("🏆 Top 5 most 'pure' occupations per RIASEC type:\n")
for code in RIASEC_CODES:
    top5 = riasec_norm.nlargest(5, code)
    print(f"  {code}:")
    for soc in top5.index:
        title = titles.get(soc, soc)
        score = riasec_norm.loc[soc, code]
        print(f"    {score:.3f}  {title}")
    print()


🏆 Top 5 most 'pure' occupations per RIASEC type:

  Realistic:
    0.464  Roustabouts, Oil and Gas
    0.460  Cleaners of Vehicles and Equipment
    0.454  Paving, Surfacing, and Tamping Equipment Operators
    0.445  Pile Driver Operators
    0.443  Rail-Track Laying and Maintenance Equipment Operators

  Investigative:
    0.340  Data Scientists
    0.332  Microbiologists
    0.329  Bioinformatics Scientists
    0.328  Statisticians
    0.325  Mathematicians

  Artistic:
    0.339  Poets, Lyricists and Creative Writers
    0.338  Fine Artists, Including Painters, Sculptors, and Illustrators
    0.332  Musicians and Singers
    0.328  Actors
    0.327  Special Effects Artists and Animators

  Social:
    0.337  Tutors
    0.330  Mental Health and Substance Abuse Social Workers
    0.328  Teaching Assistants, Special Education
    0.328  Healthcare Social Workers
    0.325  Substance Abuse and Behavioral Disorder Counselors

  Enterprising:
    0.346  First-Line Supervisors of Non-Reta

---
## 7. Work Activities Dataset
Day-to-day activities performed in each occupation.  
Examples: "Analyzing Data", "Communicating with Supervisors", "Operating Vehicles".

**Used in:** Feature Group 3 (weighted by importance) + K-Means clustering features.


In [19]:
print(f"Shape: {wa.shape}")
print(f"Unique work activities: {wa['Element Name'].nunique()}")
print(f"\nAll work activities:")
for i, name in enumerate(sorted(wa["Element Name"].unique()), 1):
    print(f"  {i:2d}. {name}")


Shape: (73308, 15)
Unique work activities: 41

All work activities:
   1. Analyzing Data or Information
   2. Assisting and Caring for Others
   3. Coaching and Developing Others
   4. Communicating with People Outside the Organization
   5. Communicating with Supervisors, Peers, or Subordinates
   6. Controlling Machines and Processes
   7. Coordinating the Work and Activities of Others
   8. Developing Objectives and Strategies
   9. Developing and Building Teams
  10. Documenting/Recording Information
  11. Drafting, Laying Out, and Specifying Technical Devices, Parts, and Equipment
  12. Establishing and Maintaining Interpersonal Relationships
  13. Estimating the Quantifiable Characteristics of Products, Events, or Information
  14. Evaluating Information to Determine Compliance with Standards
  15. Getting Information
  16. Guiding, Directing, and Motivating Subordinates
  17. Handling and Moving Objects
  18. Identifying Objects, Actions, and Events
  19. Inspecting Equipment, S

In [20]:
# Mean importance per activity across all occupations
imp_wa = (wa[wa["Scale Name"]=="Importance"]
          .groupby("Element Name")["Data Value"].mean()
          .sort_values(ascending=False))

fig, ax = plt.subplots(figsize=(11, 7))
ax.barh(imp_wa.index[::-1], imp_wa.values[::-1],
        color=plt.cm.Oranges(np.linspace(0.4, 0.9, len(imp_wa))))
ax.set_title("Mean Importance of Each Work Activity (Across All Occupations)",
             fontweight="bold")
ax.set_xlabel("Mean Importance Score (1–5)")
ax.grid(True, alpha=0.3, axis="x")
plt.tight_layout(); plt.show()


---
## 8. Job Zones Dataset
Job Zones group occupations by required **education and experience level**.

| Zone | Preparation Needed | Examples |
|------|-------------------|---------|
| 1 | Little or none | Dishwashers, Parking Lot Attendants |
| 2 | Some | Electricians, Truck Drivers |
| 3 | Medium | Dental Assistants, Police Officers |
| 4 | Considerable | Accountants, Teachers |
| 5 | Extensive | Doctors, Lawyers, Scientists |

**Used in:** Career Transition Path (measuring how many zones apart two careers are).


In [21]:
print(f"Shape: {jz.shape}")
print(f"\nColumns: {jz.columns.tolist()}")
print(f"\nJob Zone value counts:")
print(jz["Job Zone"].value_counts().sort_index())


Shape: (923, 5)

Columns: ['O*NET-SOC Code', 'Title', 'Job Zone', 'Date', 'Domain Source']

Job Zone value counts:
Job Zone
2    331
3    213
4    225
5    154
Name: count, dtype: int64


In [22]:
zone_counts = jz["Job Zone"].value_counts().sort_index()
zone_labels = {
    1: "Zone 1\nLittle prep",
    2: "Zone 2\nSome prep",
    3: "Zone 3\nMedium prep",
    4: "Zone 4\nConsiderable",
    5: "Zone 5\nExtensive"
}

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(
    [zone_labels.get(k, str(k)) for k in zone_counts.index],
    zone_counts.values,
    color=["#2ecc71","#3498db","#f39c12","#e74c3c","#9b59b6"],
    alpha=0.85, edgecolor="white", width=0.6
)
ax.set_title("Number of Occupations per Job Zone (Education/Experience Level)",
             fontweight="bold")
ax.set_ylabel("Number of Occupations")
for bar, val in zip(bars, zone_counts.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+2,
            str(val), ha="center", fontweight="bold")
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout(); plt.show()


---
## 9. Task Statements Dataset
Free-text descriptions of specific tasks performed in each occupation.  
Example: *"Analyze data to identify trends and draw conclusions."*

**Used in:** TF-IDF vectorization → Feature Group 2 (text features).  
One occupation can have 5–30 task statements. We concatenate them into one document per occupation.


In [23]:
print(f"Shape: {ts.shape}")
print(f"Columns: {ts.columns.tolist()}")
print(f"Unique occupations with tasks: {ts['O*NET-SOC Code'].nunique()}")

# How many tasks per occupation?
tasks_per_occ = ts.groupby("O*NET-SOC Code").size()
print(f"\nTasks per occupation:")
print(tasks_per_occ.describe().round(1))

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(tasks_per_occ, bins=40, color="#1abc9c", alpha=0.85, edgecolor="white")
ax.set_title("Distribution: How Many Tasks per Occupation?", fontweight="bold")
ax.set_xlabel("Number of Task Statements")
ax.set_ylabel("Number of Occupations")
ax.axvline(tasks_per_occ.mean(), color="red", linestyle="--",
           label=f"Mean = {tasks_per_occ.mean():.1f}")
ax.legend()
plt.tight_layout(); plt.show()


Shape: (18796, 8)
Columns: ['O*NET-SOC Code', 'Title', 'Task ID', 'Task', 'Task Type', 'Incumbents Responding', 'Date', 'Domain Source']
Unique occupations with tasks: 923

Tasks per occupation:
count   923.000
mean     20.400
std       6.300
min       4.000
25%      16.000
50%      20.000
75%      25.000
max      40.000
dtype: float64


In [24]:
# Sample task statements for a specific occupation
sample_soc = ts["O*NET-SOC Code"].iloc[0]
sample_title = occ.set_index("O*NET-SOC Code").loc[sample_soc, "Title"]
task_col = "Task" if "Task" in ts.columns else ts.columns[-1]

print(f"Sample task statements for: {sample_title} ({sample_soc})\n")
sample_tasks = ts[ts["O*NET-SOC Code"]==sample_soc][task_col].tolist()
for i, t in enumerate(sample_tasks[:8], 1):
    print(f"  {i}. {t}")


Sample task statements for: Chief Executives (11-1011.00)

  1. Direct or coordinate an organization's financial or budget activities to fund operations, maximize investments, or increase efficiency.
  2. Confer with board members, organization officials, or staff members to discuss issues, coordinate activities, or resolve problems.
  3. Prepare budgets for approval, including those for funding or implementation of programs.
  4. Direct, plan, or implement policies, objectives, or activities of organizations or businesses to ensure continuing operations, to maximize returns on investments, or to increase productivity.
  5. Prepare or present reports concerning activities, expenses, budgets, government statutes or rulings, or other items affecting businesses or program services.
  6. Implement corrective action plans to solve organizational or departmental problems.
  7. Analyze operations to evaluate performance of a company or its staff in meeting objectives or to determine areas of 

---
## 10. Missing Value Analysis Across All Files


In [25]:
print(f"{'Dataset':<25} {'Total Cells':>12} {'Missing':>10} {'%':>8}  Imputation Strategy")
print("─"*80)

strategies = {
    "Occupation Data":  "None needed (0% missing)",
    "Skills":           "Fill 0 — skill absent = not required for this job",
    "Abilities":        "Fill 0 — ability absent = not required for this job",
    "Knowledge":        "Fill 0 — domain not applicable to this job",
    "Interests":        "None needed (0% missing)",
    "Work Activities":  "Fill 0 — activity not performed in this job",
    "Work Styles":      "None needed (0% missing)",
    "Job Zones":        "None needed (0% missing)",
    "Task Statements":  "Drop rows; empty string → sparse TF-IDF vector",
}

for name, df in datasets.items():
    total   = df.size
    missing = df.isnull().sum().sum()
    pct     = missing / total * 100
    strat   = strategies.get(name, "Fill 0")
    print(f"  {name:<23} {total:>12,} {missing:>10,} {pct:>7.2f}%  {strat}")

print("─"*80)
print(f"\n💡 We fill ALL numeric missings with 0, NOT median.")
print(f"   Reason: a missing O*NET score means the skill/activity is")
print(f"   irrelevant to that occupation — not that the value is unknown.")
print(f"   Filling with median would falsely give every job a 'medium'")
print(f"   score in skills they genuinely never use.")


Dataset                    Total Cells    Missing        %  Imputation Strategy
────────────────────────────────────────────────────────────────────────────────
  Occupation Data                4,064          0    0.00%  None needed (0% missing)
  Skills                       938,700     31,290    3.33%  Fill 0 — skill absent = not required for this job
  Abilities                  1,394,640     46,488    3.33%  Fill 0 — ability absent = not required for this job
  Knowledge                    885,060     90,608   10.24%  Fill 0 — domain not applicable to this job
  Interests                     74,763          0    0.00%  None needed (0% missing)
  Work Activities            1,099,620    110,900   10.09%  Fill 0 — activity not performed in this job
  Work Styles                  336,798          0    0.00%  None needed (0% missing)
  Job Zones                      4,615          0    0.00%  None needed (0% missing)
  Task Statements              150,368      2,035    1.35%  Drop rows;

---
## 11. How All Datasets Link Together

All 9 files share the `O*NET-SOC Code` as the primary key.  
After loading, we pivot each file from long-format → wide-format,  
then join them all on SOC code to build the final feature matrix.


In [26]:
# Show how many occupations each file covers
print("SOC code coverage per dataset:")
print("-"*40)
for name, df in datasets.items():
    if "O*NET-SOC Code" in df.columns:
        n = df["O*NET-SOC Code"].nunique()
        print(f"  {name:<25}: {n:,} unique occupations")

print(f"\n📌 After inner-joining all files, we get the occupations")
print(f"   that appear in ALL datasets — this is our final ML dataset.")

# Simulate what the join gives us
sets = [set(df["O*NET-SOC Code"].unique())
        for df in datasets.values()
        if "O*NET-SOC Code" in df.columns]
common = set.intersection(*sets)
print(f"   Occupations in ALL 9 files: {len(common):,}")


SOC code coverage per dataset:
----------------------------------------
  Occupation Data          : 1,016 unique occupations
  Skills                   : 894 unique occupations
  Abilities                : 894 unique occupations
  Knowledge                : 894 unique occupations
  Interests                : 923 unique occupations
  Work Activities          : 894 unique occupations
  Work Styles              : 891 unique occupations
  Job Zones                : 923 unique occupations
  Task Statements          : 923 unique occupations

📌 After inner-joining all files, we get the occupations
   that appear in ALL datasets — this is our final ML dataset.
   Occupations in ALL 9 files: 862


In [27]:
# Final feature matrix overview (what the model actually sees)
lines = [
    "=" * 62,
    "  FINAL FEATURE MATRIX STRUCTURE",
    "=" * 62,
    "  Group 1: Skills + Abilities + Knowledge + Styles",
    "            prefix: skill__, abil__, know__, style__",
    "            O*NET Level scores 0-7",
    "            Imputation: fill 0 (absent = not required)",
    "",
    "  Group 2: Task Statements -> TF-IDF",
    "            prefix: tfidf__",
    "            100 numeric features via TF-IDF",
    "            Imputation: empty string -> all-zero row",
    "",
    "  Group 3: Work Activities (Importance scale)",
    "            prefix: wa__",
    "            Importance score 1-5",
    "            Imputation: fill 0 (activity not performed)",
    "=" * 62,
    "  TARGET: RIASEC scores (6 columns, normalized to sum=1)",
    "          Multi-output regression -> predict all 6 at once",
    "=" * 62,
]
print("\n".join(lines))


  FINAL FEATURE MATRIX STRUCTURE
  Group 1: Skills + Abilities + Knowledge + Styles
            prefix: skill__, abil__, know__, style__
            O*NET Level scores 0-7
            Imputation: fill 0 (absent = not required)

  Group 2: Task Statements -> TF-IDF
            prefix: tfidf__
            100 numeric features via TF-IDF
            Imputation: empty string -> all-zero row

  Group 3: Work Activities (Importance scale)
            prefix: wa__
            Importance score 1-5
            Imputation: fill 0 (activity not performed)
  TARGET: RIASEC scores (6 columns, normalized to sum=1)
          Multi-output regression -> predict all 6 at once


---
## 12. EDA Summary & Key Takeaways

| Finding | Implication for ML |
|---------|-------------------|
| RIASEC scores are continuous [0–1] | → Use regression, NOT classification |
| Investigative and Conventional are most correlated (−0.35) | → They rarely appear together in same career |
| Artistic has high variance | → Strongly separates careers |
| Knowledge has 10% missing | → Fill with 0, not median |
| Each occupation has avg ~12 task statements | → TF-IDF will have decent text signal |
| Job Zone 3 has the most occupations | → Dataset is realistic-world distribution |
| 83/282 features typically matched from user skills | → Synonym map working correctly |

**Bottom line:** The O*NET database gives us a rich, multi-dimensional view of every occupation.  
By combining skills (what you can do), interests (what you enjoy), and task text (what the job involves),  
we can build a highly accurate career matching system.
